# Inheritance Mechanics, Superclass Construction, and Abstract Base Classes

**Session:** 2026-11-01 · OOP with Python  
**Module:** 01 · Python Fundamentals & Algorithmic Thinking

Inheritance is one of the foundational pillars of Object-Oriented Programming (OOP). It formalizes an **"is-a"** relationship between classes, enabling a derived subclass to inherit state and behavior from a base superclass. This mechanism eliminates redundant code, promotes modularity, and establishes hierarchical taxonomies for extensible system design.

## Learning Outcomes

- Understand the motivation for inheritance by contrasting duplicate class definitions with shared base abstractions.
- Master single inheritance syntax and constructor chaining via `super().__init__()`.
- Understand method resolution order (MRO) and the C3 linearization algorithm in multiple inheritance.
- Define formal contracts and interface specifications using Python's `abc.ABC` and `@abstractmethod`.
- Load and parse heterogeneous geometric shape models polymorphically from structured datasets.

## 1. The Need for Inheritance: Code Reusability vs Redundancy

Consider two utility classes, `Math1` and `Math2`. `Math1` provides elementary operations: checking for even numbers and computing factorials:
$$\text{factorial}(n) = n! = \prod_{k=1}^n k$$

`Math2` requires all capabilities of `Math1` plus the estimation of Euler's number $e$ via Taylor series expansion:
$$e = \sum_{k=0}^{\infty} \frac{1}{k!} = 1 + \frac{1}{1!} + \frac{1}{2!} + \frac{1}{3!} + \dots$$

Without inheritance, `Math2` must duplicate the methods of `Math1`. If a bug is fixed or an optimization is added to `factorial()`, changes must be made in multiple places.

In [ ]:
# Naive approach: Duplicated implementation without inheritance
class Math1:
    """Elementary mathematical operations."""

    def is_even(self, number: int) -> bool:
        return number % 2 == 0

    def factorial(self, number: int) -> int:
        if number < 0:
            raise ValueError("Factorial is undefined for negative integers.")
        result = 1
        for i in range(1, number + 1):
            result *= i
        return result


class DuplicateMath2:
    """Anti-pattern: Duplicating Math1 logic manually."""

    def is_even(self, number: int) -> bool:
        return number % 2 == 0

    def factorial(self, number: int) -> int:
        if number < 0:
            raise ValueError("Factorial is undefined for negative integers.")
        result = 1
        for i in range(1, number + 1):
            result *= i
        return result

    def estimate_euler(self, iterations: int) -> float:
        """Estimate Euler's number using duplicated factorial."""
        return 1.0 + sum(1.0 / self.factorial(i) for i in range(1, iterations + 1))


math1 = Math1()
dup_math2 = DuplicateMath2()

print("Math1 is_even(6):", math1.is_even(6))
print("Math1 factorial(5):", math1.factorial(5))
print("DuplicateMath2 estimate_euler(10):", dup_math2.estimate_euler(10))

## 2. Refactoring with Inheritance and `super()`

In Python, a subclass inherits from a superclass by specifying the parent class in parentheses: `class Subclass(Superclass):`.

To call methods from the parent class explicitly—especially inside overridden methods or extended functionality—we use the built-in `super()` function. `super()` returns a proxy object that delegates method calls to the parent or sibling classes in the Method Resolution Order (MRO).

In [ ]:
class Math2(Math1):
    """Refactored Math2 inheriting from Math1."""

    def estimate_euler(self, iterations: int) -> float:
        """Estimate Euler's number by reusing the inherited factorial method."""
        if iterations < 1:
            raise ValueError("Iterations must be at least 1.")
        # super().factorial(i) delegates directly to Math1.factorial
        return 1.0 + sum(1.0 / super().factorial(i) for i in range(1, iterations + 1))


math2 = Math2()

# Inherited methods
assert math2.is_even(10) is True
assert math2.factorial(4) == 24

# New specialized method
e_approx = math2.estimate_euler(10)
print(f"Euler's number approximation (10 terms): {e_approx:.8f}")
print("True e value from math.e:               2.71828183")

## 3. Superclass Construction and Domain Taxonomies

When modeling real-world domain entities, subclasses often require additional state attributes alongside the attributes of the parent class.

Constructor chaining ensures that the superclass initializes its own internal state before the subclass initializes specialized fields. This is accomplished by calling `super().__init__(...)` within the subclass constructor.

In [ ]:
class Animal:
    """Base class representing an animal in a taxonomic hierarchy."""

    def __init__(self, name: str, species: str, age: int) -> None:
        self.name = name
        self.species = species
        self.age = age

    def describe(self) -> str:
        return f"[{self.species}] {self.name}, Age: {self.age}"

    def make_sound(self) -> str:
        raise NotImplementedError("Subclasses must implement make_sound()")


class Dog(Animal):
    """Concrete animal subclass representing a canine."""

    def __init__(self, name: str, age: int, breed: str, owner: str) -> None:
        super().__init__(name=name, species="Canis familiaris", age=age)
        self.breed = breed
        self.owner = owner

    def make_sound(self) -> str:
        return "Woof! Woof!"

    def describe(self) -> str:
        base_desc = super().describe()
        return f"{base_desc} (Breed: {self.breed}, Owner: {self.owner})"


class Cat(Animal):
    """Concrete animal subclass representing a feline."""

    def __init__(self, name: str, age: int, coat_color: str, indoor: bool = True) -> None:
        super().__init__(name=name, species="Felis catus", age=age)
        self.coat_color = coat_color
        self.indoor = indoor

    def make_sound(self) -> str:
        return "Meow!"

    def describe(self) -> str:
        base_desc = super().describe()
        mode = "Indoor" if self.indoor else "Outdoor"
        return f"{base_desc} (Coat: {self.coat_color}, Mode: {mode})"


dog = Dog(name="Barnaby", age=4, breed="Golden Retriever", owner="Dr. Eleanor Vance")
cat = Cat(name="Cleo", age=2, coat_color="Tortoiseshell", indoor=True)

print(dog.describe(), "-> Sound:", dog.make_sound())
print(cat.describe(), "-> Sound:", cat.make_sound())

## 4. Method Overriding in Enterprise Compensation Models

Method overriding occurs when a subclass provides a specialized implementation of a method that is already defined in its superclass.

A classic example is corporate payroll: a standard `Employee` receives a base salary, while a `Manager` receives a base salary plus an annual bonus. By calling `super().compute_annual_salary()`, the manager class reuses the base calculation logic while adding bonus logic.

In [ ]:
class Employee:
    """Standard organizational employee."""

    def __init__(self, name: str, employee_id: str, monthly_salary: float) -> None:
        self._name = name
        self._employee_id = employee_id
        self._monthly_salary = float(monthly_salary)

    def compute_annual_salary(self) -> float:
        """Annual compensation based on 12 regular monthly pay periods."""
        return self._monthly_salary * 12.0

    def get_summary(self) -> str:
        return f"Employee #{self._employee_id}: {self._name} | Annual: ${self.compute_annual_salary():,.2f}"


class Manager(Employee):
    """Managerial employee with performance bonus compensation."""

    def __init__(
        self, name: str, employee_id: str, monthly_salary: float, annual_bonus: float
    ) -> None:
        super().__init__(name, employee_id, monthly_salary)
        self._annual_bonus = float(annual_bonus)

    def compute_annual_salary(self) -> float:
        """Annual compensation extending the base salary with a bonus."""
        base_annual = super().compute_annual_salary()
        return base_annual + self._annual_bonus

    def get_summary(self) -> str:
        base_annual = super().compute_annual_salary()
        return (
            f"Manager #{self._employee_id}: {self._name} | "
            f"Base: ${base_annual:,.2f} + Bonus: ${self._annual_bonus:,.2f} = "
            f"Total: ${self.compute_annual_salary():,.2f}"
        )


engineer = Employee(name="Marcus Brody", employee_id="E-101", monthly_salary=6500.0)
director = Manager(
    name="Sophia Zhang", employee_id="M-202", monthly_salary=9500.0, annual_bonus=24000.0
)

print(engineer.get_summary())
print(director.get_summary())

assert engineer.compute_annual_salary() == 78000.0
assert director.compute_annual_salary() == 138000.0

## 5. Abstract Base Classes (ABCs) and Interface Enforcement

Python provides the `abc` module to formalize class interfaces through **Abstract Base Classes**.
- Classes inheriting from `abc.ABC` cannot be instantiated directly if they contain any `@abstractmethod` decorators.
- Subclasses must implement all abstract methods before they can be instantiated.
- This provides compile-time-like interface guarantees in Python's dynamic runtime.

In [ ]:
import math
from abc import ABC, abstractmethod


class Shape(ABC):
    """Abstract base class for two-dimensional geometric figures."""

    def __init__(self, shape_id: str, color: str = "transparent") -> None:
        self.shape_id = shape_id
        self.color = color

    @abstractmethod
    def area(self) -> float:
        """Compute and return the surface area."""
        pass

    @abstractmethod
    def perimeter(self) -> float:
        """Compute and return the boundary perimeter."""
        pass

    def compute_mass(self, density_g_cm2: float) -> float:
        """Concrete shared method computing mass based on surface density."""
        return self.area() * density_g_cm2


# Attempting to instantiate an abstract class directly raises TypeError:
try:
    s = Shape("SHP-TEST")
except TypeError as e:
    print("Direct instantiation guard verified:", e)

In [ ]:
class Circle(Shape):
    """Concrete circle shape defined by radius r."""

    def __init__(self, shape_id: str, radius: float, color: str = "transparent") -> None:
        super().__init__(shape_id, color)
        if radius <= 0:
            raise ValueError("Radius must be strictly positive.")
        self.radius = float(radius)

    def area(self) -> float:
        return math.pi * (self.radius**2)

    def perimeter(self) -> float:
        return 2.0 * math.pi * self.radius


class Rectangle(Shape):
    """Concrete rectangle shape defined by width and height."""

    def __init__(
        self, shape_id: str, width: float, height: float, color: str = "transparent"
    ) -> None:
        super().__init__(shape_id, color)
        if width <= 0 or height <= 0:
            raise ValueError("Width and height must be positive.")
        self.width = float(width)
        self.height = float(height)

    def area(self) -> float:
        return self.width * self.height

    def perimeter(self) -> float:
        return 2.0 * (self.width + self.height)


class Square(Rectangle):
    """Square as a specialized rectangle where width == height."""

    def __init__(self, shape_id: str, side: float, color: str = "transparent") -> None:
        super().__init__(shape_id=shape_id, width=side, height=side, color=color)
        self.side = float(side)


# Test implementations
circ = Circle("C-1", radius=5.0, color="sapphire")
rect = Rectangle("R-1", width=8.0, height=4.5, color="crimson")
sq = Square("S-1", side=6.0, color="emerald")

print(f"Circle area: {circ.area():.4f}, perimeter: {circ.perimeter():.4f}")
print(f"Rectangle area: {rect.area():.4f}, perimeter: {rect.perimeter():.4f}")
print(f"Square area: {sq.area():.4f}, perimeter: {sq.perimeter():.4f}")

## 6. Parsing Structured Shape Datasets Polymorphically

Let's load `data/shapes_collection.json` and instantiate the corresponding domain classes dynamically using a factory pattern.

In [ ]:
import json
from pathlib import Path

data_file = Path("../data/shapes_collection.json")
if not data_file.exists():
    data_file = Path("data/shapes_collection.json")

with data_file.open("r", encoding="utf-8") as f:
    raw_specs = json.load(f)


def shape_factory(spec: dict) -> Shape:
    stype = spec["type"]
    params = spec["parameters"]
    sid = spec["id"]
    color = spec["color"]

    if stype == "circle":
        return Circle(shape_id=sid, radius=params["radius"], color=color)
    elif stype == "rectangle":
        return Rectangle(shape_id=sid, width=params["width"], height=params["height"], color=color)
    elif stype == "square":
        return Square(shape_id=sid, side=params["side"], color=color)
    elif stype == "triangle":
        # Concrete Triangle inline adapter
        class Triangle(Shape):
            def __init__(self, shape_id, base, height, side_a, side_b, side_c, color):
                super().__init__(shape_id, color)
                self.base = base
                self.height = height
                self.side_a = side_a
                self.side_b = side_b
                self.side_c = side_c

            def area(self) -> float:
                return 0.5 * self.base * self.height

            def perimeter(self) -> float:
                return self.side_a + self.side_b + self.side_c

        return Triangle(
            sid,
            params["base"],
            params["height"],
            params["side_a"],
            params["side_b"],
            params["side_c"],
            color,
        )
    else:
        raise ValueError(f"Unknown shape type: {stype}")


shapes = [shape_factory(s) for s in raw_specs]
print(f"Successfully instantiated {len(shapes)} polymorphic shapes:")
for shape in shapes:
    print(
        f"  Shape {shape.shape_id:8s} | Class: {shape.__class__.__name__:10s} | Area: {shape.area():8.2f} | Perimeter: {shape.perimeter():8.2f}"
    )

## 7. Multiple Inheritance and Method Resolution Order (MRO)

Python supports multiple inheritance, allowing a class to inherit from more than one parent class: `class Child(Base1, Base2):`.

To resolve method calls without ambiguity—especially in diamond inheritance hierarchies—Python employs the **C3 Linearization** algorithm. The sequence of classes searched during attribute lookup is stored in `Class.__mro__`.

In [ ]:
class A:
    def execute(self) -> str:
        return "A.execute"


class B(A):
    def execute(self) -> str:
        return f"B -> {super().execute()}"


class C(A):
    def execute(self) -> str:
        return f"C -> {super().execute()}"


class D(B, C):
    def execute(self) -> str:
        return f"D -> {super().execute()}"


instance_d = D()
print("Execution dispatch chain:")
print(instance_d.execute())

print("\nMethod Resolution Order (MRO) for class D:")
for idx, cls in enumerate(D.__mro__):
    print(f"  Step {idx + 1}: {cls.__name__}")